In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
sns.set(style="whitegrid")

In [3]:

# -------------------------------------------------------
# 0. GENERATE SYNTHETIC RAW DATASET (simulates a messy real-world CSV)
# -------------------------------------------------------
n = 1000
brands = ["Toyota", "Honda", "Ford", "BMW", "Hyundai", "Suzuki", "Kia"]
fuel_types = ["Petrol", "Diesel", "Hybrid", "Electric"]
transmissions = ["Manual", "Automatic"]

brand = np.random.choice(brands, n, p=[0.22, 0.18, 0.15, 0.1, 0.15, 0.12, 0.08])
year = np.random.randint(2005, 2024, n)
age = 2024 - year
mileage = (age * np.random.normal(12000, 3000, n)).clip(1000, None)
engine_size = np.round(np.random.uniform(1.0, 4.0, n), 1)
fuel_type = np.random.choice(fuel_types, n, p=[0.55, 0.25, 0.15, 0.05])
transmission = np.random.choice(transmissions, n, p=[0.4, 0.6])
owners = np.random.choice([1, 2, 3, 4], n, p=[0.5, 0.3, 0.15, 0.05])

brand_premium = {"BMW": 12000, "Toyota": 3000, "Honda": 2500, "Ford": 1500,
                  "Hyundai": 1000, "Kia": 800, "Suzuki": 500}
fuel_premium = {"Electric": 6000, "Hybrid": 3000, "Diesel": 1000, "Petrol": 0}

price = (
    20000
    + np.array([brand_premium[b] for b in brand])
    + np.array([fuel_premium[f] for f in fuel_type])
    + engine_size * 1500
    - age * 900
    - mileage * 0.05
    - owners * 700
    + np.where(transmission == "Automatic", 1200, 0)
    + np.random.normal(0, 1500, n)
).clip(1500, None)

df = pd.DataFrame({
    "brand": brand,
    "year": year,
    "mileage": mileage.round(0),
    "engine_size": engine_size,
    "fuel_type": fuel_type,
    "transmission": transmission,
    "owners": owners,
    "price": price.round(0),
})

# Inject realistic messiness
missing_idx = np.random.choice(df.index, size=40, replace=False)
df.loc[missing_idx, "mileage"] = np.nan
missing_idx2 = np.random.choice(df.index, size=25, replace=False)
df.loc[missing_idx2, "engine_size"] = np.nan

# Inconsistent text casing
df.loc[df.sample(50, random_state=1).index, "fuel_type"] = df["fuel_type"].str.upper()
df.loc[df.sample(50, random_state=2).index, "transmission"] = df["transmission"].str.lower()

# Outliers
outlier_idx = np.random.choice(df.index, size=10, replace=False)
df.loc[outlier_idx, "price"] = df.loc[outlier_idx, "price"] * 6

# Duplicate rows
df = pd.concat([df, df.sample(15, random_state=3)], ignore_index=True)

df.to_csv(r"C:\Users\user\Desktop\iris\car_data_raw.csv", index=False)

print("=" * 60)
print("RAW DATA LOADED")
print("=" * 60)
print(f"Shape: {df.shape}")
print(df.head())

RAW DATA LOADED
Shape: (1015, 8)
    brand  year   mileage  engine_size fuel_type transmission  owners    price
0  Suzuki  2009       NaN          3.0    Petrol    Automatic       1   3092.0
1   Honda  2021   41367.0          2.8    Hybrid       Manual       2  23662.0
2     Kia  2008  237903.0          1.1    Petrol    Automatic       3   1500.0
3     BMW  2017  115072.0          3.6    Diesel    Automatic       1  29497.0
4     Kia  2020       NaN          2.3    Petrol       Manual       1  17467.0


In [4]:

# -------------------------------------------------------
# 1. DATA CLEANING
# -------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 1: DATA CLEANING")
print("=" * 60)

print("\nMissing values before cleaning:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Standardize text columns
df["fuel_type"] = df["fuel_type"].str.strip().str.capitalize()
df["transmission"] = df["transmission"].str.strip().str.capitalize()

# Drop duplicates
df = df.drop_duplicates().reset_index(drop=True)

# Fill missing numeric values with median
for col in ["mileage", "engine_size"]:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Remove price outliers using IQR method
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
before = df.shape[0]
df = df[(df["price"] >= lower) & (df["price"] <= upper)].reset_index(drop=True)
print(f"\nOutliers removed from 'price': {before - df.shape[0]} rows")

# Feature engineering: car age instead of raw year
df["car_age"] = 2024 - df["year"]
df = df.drop(columns=["year"])

print(f"\nCleaned dataset shape: {df.shape}")
print("Missing values after cleaning:")
print(df.isnull().sum())
# df.to_csv("/mnt/user-data/outputs/car_data_cleaned.csv", index=False)

# EDA visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df["price"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Price Distribution (After Cleaning)")
sns.scatterplot(x="mileage", y="price", hue="fuel_type", data=df, ax=axes[1])
axes[1].set_title("Mileage vs Price")
plt.tight_layout()
# plt.savefig("/mnt/user-data/outputs/eda_price_mileage.png", dpi=150)
plt.close()


STEP 1: DATA CLEANING

Missing values before cleaning:
brand            0
year             0
mileage         40
engine_size     26
fuel_type        0
transmission     0
owners           0
price            0
dtype: int64

Duplicate rows: 15

Outliers removed from 'price': 7 rows

Cleaned dataset shape: (993, 8)
Missing values after cleaning:
brand           0
mileage         0
engine_size     0
fuel_type       0
transmission    0
owners          0
price           0
car_age         0
dtype: int64


In [5]:
# -------------------------------------------------------
# 2. FEATURE SELECTION
# -------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 2: FEATURE SELECTION")
print("=" * 60)

X = df.drop(columns=["price"])
y = df["price"]

numeric_features = ["mileage", "engine_size", "owners", "car_age"]
categorical_features = ["brand", "fuel_type", "transmission"]

# Correlation of numeric features with target
corr_with_price = df[numeric_features + ["price"]].corr()["price"].drop("price")
print("\nCorrelation of numeric features with price:")
print(corr_with_price.sort_values(ascending=False))

plt.figure(figsize=(6, 4))
corr_with_price.sort_values().plot(kind="barh", color="teal")
plt.title("Feature Correlation with Price")
plt.tight_layout()
# plt.savefig("/mnt/user-data/outputs/feature_correlation.png", dpi=150)
plt.close()

# Preprocessing pipeline: scale numeric, one-hot encode categorical
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
])

# SelectKBest on numeric features (informational — shows statistical relevance)
selector = SelectKBest(score_func=f_regression, k="all")
selector.fit(df[numeric_features], y)
scores = pd.Series(selector.scores_, index=numeric_features).sort_values(ascending=False)
print("\nSelectKBest F-scores (numeric features):")
print(scores)

print("\nFinal features used for modeling:")
print(f"Numeric: {numeric_features}")
print(f"Categorical (one-hot encoded): {categorical_features}")


STEP 2: FEATURE SELECTION

Correlation of numeric features with price:
engine_size    0.137504
owners        -0.081684
mileage       -0.813961
car_age       -0.880198
Name: price, dtype: float64

SelectKBest F-scores (numeric features):
car_age        3408.528319
mileage        1945.585485
engine_size      19.098165
owners            6.656597
dtype: float64

Final features used for modeling:
Numeric: ['mileage', 'engine_size', 'owners', 'car_age']
Categorical (one-hot encoded): ['brand', 'fuel_type', 'transmission']


In [6]:

# -------------------------------------------------------
# 3. REGRESSION MODEL
# -------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 3: TRAINING REGRESSION MODELS")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200, random_state=42),
}

results = {}
for name, reg in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", reg)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    results[name] = {"pipeline": pipe, "preds": preds, "MAE": mae, "RMSE": rmse, "R2": r2}


STEP 3: TRAINING REGRESSION MODELS
Training samples: 794, Testing samples: 199


In [7]:
# -------------------------------------------------------
# 4. EVALUATION (MAE, RMSE)
# -------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 4: MODEL EVALUATION")
print("=" * 60)

for name, res in results.items():
    print(f"\n{name}:")
    print(f"  MAE  : ${res['MAE']:,.2f}")
    print(f"  RMSE : ${res['RMSE']:,.2f}")
    print(f"  R2   : {res['R2']:.4f}")

best_model_name = min(results, key=lambda k: results[k]["RMSE"])
print(f"\nBest model (lowest RMSE): {best_model_name}")

# Actual vs Predicted plot for best model
best_preds = results[best_model_name]["preds"]
plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_preds, alpha=0.5, color="darkorange")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "k--", lw=2)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title(f"Actual vs Predicted Price — {best_model_name}")
plt.tight_layout()
# plt.savefig("/mnt/user-data/outputs/actual_vs_predicted.png", dpi=150)
plt.close()

# Model comparison bar chart
comparison_df = pd.DataFrame({k: {"MAE": v["MAE"], "RMSE": v["RMSE"]} for k, v in results.items()}).T
comparison_df.plot(kind="bar", figsize=(7, 4.5), color=["#4C72B0", "#DD8452"])
plt.title("Model Comparison: MAE vs RMSE")
plt.ylabel("Error ($)")
plt.xticks(rotation=0)
plt.tight_layout()
# plt.savefig("/mnt/user-data/outputs/model_comparison.png", dpi=150)
plt.close()

print("\nSaved plots:")
print(" - eda_price_mileage.png")
print(" - feature_correlation.png")
print(" - actual_vs_predicted.png")
print(" - model_comparison.png")

print("\n" + "=" * 60)
print("PROJECT COMPLETE ✅")
print("=" * 60)


STEP 4: MODEL EVALUATION

Linear Regression:
  MAE  : $1,376.98
  RMSE : $1,723.89
  R2   : 0.9586

Random Forest Regressor:
  MAE  : $2,312.86
  RMSE : $3,245.48
  R2   : 0.8534

Best model (lowest RMSE): Linear Regression

Saved plots:
 - eda_price_mileage.png
 - feature_correlation.png
 - actual_vs_predicted.png
 - model_comparison.png

PROJECT COMPLETE ✅
